#### >> Utilizando a extensão jupyter (.ipynb)

In [97]:
import os
import polars as pl 
import numpy as np
import matplotlib.pyplot as plt 

pl.Config.set_tbl_rows(-1)
pl.Config.set_decimal_separator(',')
pl.Config.set_thousands_separator('.')
pl.Config.set_float_precision(2)

ENDERECO_DADOS = './../../DADOS/BOLSA_FAMILIA/'
ENDERECO_VOTACAO = './../../DADOS/VOTACAO/'

##### >> Obtendo os dados do arquivo

In [98]:

try:
    print('Lendo o arquivo...')

    df_bolsa_familia = pl.scan_parquet(ENDERECO_DADOS + 'bolsa_familia.parquet')   
    # print(df_bolsa_familia.head(5).collect())   

    df_dados_votacao = pl.read_csv(ENDERECO_VOTACAO + 'votacao_secao_2022_BR.csv', separator=';', encoding='iso-8859-1')
    # print(df_dados_votacao)
    # print(df_votacao.describe())
    # for coluna in df_dados_votacao.columns:
    #     print(coluna)
    
except Exception as e:
    print(f'Erro ao ler o arquivo: {e}')

Lendo o arquivo...


#### >> Filtrar p/segundo turno 'NR_TURNO' e 'NR_VOTAVEL' 13 e 22

In [99]:

try:
    print('Filtrando os dados para o segundo turno e candidatos 13 e 22...')
    df_turno02 = df_dados_votacao.filter(
    (pl.col('NR_TURNO') == 2) & 
    (pl.col('NR_VOTAVEL').is_in([13, 22]))
    )

    print('Dados filtrados com sucesso!')

except Exception as e:
    print(f'Erro ao filtrar os dados: {e}')

Filtrando os dados para o segundo turno e candidatos 13 e 22...
Dados filtrados com sucesso!


#### Processando Votação 

In [100]:
try:    
    print('Agrupando os dados...')

    # >> Delimitar  as variaveis 'SG_UF', 'NM_VOTAVEL', 'QT_VOTOS'    
    df_votacao =df_turno02.lazy().select(['SG_UF', 'NM_VOTAVEL', 'QT_VOTOS'])
    # print(df_votacao.head(5))

    # >> Converter para Categorical
    df_votacao = df_votacao.with_columns([
        pl.col('SG_UF').cast(pl.Categorical),
        pl.col('NM_VOTAVEL').cast(pl.Categorical)
    ])
    # print(df_votacao.head(5))

    # Agrupar/totalizar
    df_votacao = (
    df_votacao
    .group_by(["SG_UF", "NM_VOTAVEL"])
    .agg(
        pl.col("QT_VOTOS").sum()
    )
    .sort("SG_UF")
    )

    # coletar os dados
    df_votacao = df_votacao.collect()
    # display(df_votacao) # >> print do Jupyter
    # print(df_votacao.head(5)) # >> print do terminal 

except Exception as e:
    print(f'Erro ao agrupar os dados: {e}')

Agrupando os dados...


#### >> Processamento Bolsa Família

In [101]:
try:
    # Delimitar as variáveis, converter Categorical, Agrupar/totalizar
    print('Processando os dados de bolsa família...')

    # delimitando as vaiáveis 'UF', 'VALOR PARCELA'
    df_bolsa_familia = df_bolsa_familia.lazy().select(['UF', 'VALOR PARCELA'])

    # Converter Categorical
    df_bolsa_familia = df_bolsa_familia.with_columns(
        pl.col('UF').cast(pl.Categorical)
    )

    # Agrupar/totalizar
    df_bolsa_familia = (
        df_bolsa_familia.group_by('UF')
        .agg(pl.col('VALOR PARCELA').sum())
        .sort('UF', descending=False)
    )

    # coletar os dados
    df_bolsa_familia = df_bolsa_familia.collect()
    # display(df_bolsa_familia)  # print do jupyter

except Exception as e:
    print(f'Erro ao processar dados de bolsa família: {e}')

Processando os dados de bolsa família...


#### >> Realizando o Merge nos dados

In [102]:
try:
    print('Realizando o Merge...')
    df_votos_bolsa_familia = (
        df_votacao.join(df_bolsa_familia, left_on='SG_UF', right_on='UF')
        )
      

    df_votos_bolsa_familia = df_votos_bolsa_familia.collect().sort('SG_UF', descending=False)
    # display(df_votos_bolsa_familia)

except Exception as e:
    print(f'Erro ao realizar o Merge: {e}')

Realizando o Merge...
Erro ao realizar o Merge: 'DataFrame' object has no attribute 'collect'


#### >> Correlação de Pearson

In [103]:
try:
    print('Verificando a Correlação entre os Votos e Bolsa Família...')

    dict_correlacao = {} # guardar a correlação dos candidatos

    for candidato in df_votos_bolsa_familia['NM_VOTAVEL'].unique():
        df_candidato = df_votos_bolsa_familia.filter(pl.col('NM_VOTAVEL') == candidato)

        # Criar arrays com numpy 'VALOR PARCELA' e 'QT_VOTOS'
        array_votos = np.array(df_candidato['QT_VOTOS'])
        array_bolsa_familia = np.array(df_candidato['VALOR PARCELA'])
        
        # Calculando a correlação
        # O resultado é sempre uma matriz, onde o elemento [0, 1] é a correlação entre os dois arrays. Primeira linha e segunda coluna, ou seja, a correlação entre os votos e o valor da bolsa família.
        correlacao = np.corrcoef(array_votos, array_bolsa_familia)[0, 1]

        print(f'Correlação do candidato {candidato}: {correlacao:.4f}')

        dict_correlacao[candidato] = correlacao

        print('Correlação calculada!')
    

except Exception as e:
    print(f'Erro ao verificar a Correlação: {e}')

Verificando a Correlação entre os Votos e Bolsa Família...
Correlação do candidato JAIR MESSIAS BOLSONARO: 0.6328
Correlação calculada!
Correlação do candidato LUIZ INÁCIO LULA DA SILVA: 0.8908
Correlação calculada!


#### >> Visualização dos dados

In [ ]:
try:
    print('Plotando os gráficos...')
    plt.subplots(2, 2, figsize=(18, 10))
    plt.suptitle('Votação X Bolsa Família', fontsize=16)

    # Posição 1 >> Ranking Lula
    plt.subplot(2, 2, 1)
    df_lula = df_votos_bolsa_familia.filter(pl.col('NM_VOTAVEL') == 'LUIZ INÁCIO LULA DA SILVA')
    df_lula = df_lula.sort('QT_VOTOS', descending=False)

    # Gráfico de Colunas
    plt.bar(df_lula['SG_UF'], df_lula['QT_VOTOS'], color='red', alpha=0.7, label='Votos')   
    plt.title('Ranking Lula', fontsize=14) 

    # Posição 2 >> Ranking Bolsonaro
    plt.subplot(2, 2, 2)
    df_bolsonaro = df_votos_bolsa_familia.filter(pl.col('NM_VOTAVEL') == 'JAIR MESSIAS BOLSONARO')
    df_bolsonaro = df_bolsonaro.sort('QT_VOTOS', descending=False)

    # Gráfico de Colunas
    plt.bar(df_bolsonaro['SG_UF'], df_bolsonaro['QT_VOTOS'], color='yellow', alpha=0.7, label='Votos')
    plt.title('Ranking Bolsonaro', fontsize=14)

    # Posição 3 >> Ranking Bolsa Família
    plt.subplot(2, 2, 3)
    df_bolsa_familia = df_votos_bolsa_familia.select(['SG_UF', 'VALOR PARCELA']).unique().sort('VALOR PARCELA', descending=False)
    
    # Gráfico de Barras    
    df_bolsa_familia = df_bolsa_familia.sort('VALOR PARCELA', descending=False)
    plt.bar(df_bolsa_familia['SG_UF'], df_bolsa_familia['VALOR PARCELA'], color='gray', alpha=0.7, label='Bolsa Família')
    plt.title('Ranking Bolsa Família', fontsize=14)

    # Posição 4 >> Correlação
    plt.subplot(2, 2, 4)
    plt.title('Correlação', fontsize=14)

    x = 0.2
    y = 0.6

    for candidato, correlacao in dict_correlacao.items():
        plt.text(x, y, f'Correlação {candidato}: {correlacao:.4f}', fontsize=12)

        y = y - 0.2

    
except Exception as e:
    print(f'Erro ao gerar o gráfico: {e}')

#### >> Gráfico de dispersão